<a href="https://colab.research.google.com/github/prabhas16/project/blob/main/SmartEmotionDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install all required packages
!pip install gradio transformers torch librosa git+https://github.com/openai/whisper.git


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-hwglb6gl
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-hwglb6gl
  Resolved https://github.com/openai/whisper.git to commit 04f449b8a437f1bbd3dba5c9f826aca972e7709a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.5 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=2c75840e59824707ef40f5252933cde2050dc25710da924ba9e57e710c6ce505
  Stored in directory: /tmp/pip-ephem-wheel-cache-5un78tkr/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper


In [2]:
# Import necessary libraries
import gradio as gr
from transformers import pipeline
import whisper


In [3]:
# ================================
# Loading Pre-trained Models
# ================================
print("🔄 Loading models... This may take a minute.")

# Sentiment analysis model
sentiment_model = pipeline("sentiment-analysis")

# Emotion detection model
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

# Whisper speech-to-text model
whisper_model = whisper.load_model("small")  # "small" for speed
print("✅ Models loaded successfully!")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


🔄 Loading models... This may take a minute.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 64.1MiB/s]


✅ Models loaded successfully!


In [4]:
# ================================
# Loading Pre-trained Models
# ================================
print("🔄 Loading models... This may take a minute.")

# Determine the device to use
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    # Sentiment analysis model
    sentiment_model = pipeline("sentiment-analysis", device=device)
    print("✅ Sentiment model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load sentiment model: {e}")
    sentiment_model = None

try:
    # Emotion detection model
    emotion_model = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        device=device
    )
    print("✅ Emotion detection model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load emotion detection model: {e}")
    emotion_model = None

try:
    # Whisper speech-to-text model
    # whisper_model = whisper.load_model("small", device=device)  # "small" for speed
    # Using a smaller model for potentially faster processing on CPU
    whisper_model = whisper.load_model("base", device=device)
    print("✅ Whisper model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load Whisper model: {e}")
    whisper_model = None


# Define a mapping of emotions to therapy suggestions
emotion_therapies = {
    'joy': "Consider celebrating your positive emotions! Sharing your joy with others or engaging in activities that make you happy can enhance well-being.",
    'sadness': "It's okay to feel sad. Talking to a friend, family member, or therapist can help. Engaging in self-care activities like mindfulness or gentle exercise can also be beneficial.",
    'anger': "Finding healthy ways to manage anger is important. Techniques like deep breathing, journaling, or physical activity can help. If anger is persistent, consider talking to a professional.",
    'fear': "Facing fears can be challenging, but also empowering. Techniques like gradual exposure or cognitive restructuring with a therapist can be helpful. Mindfulness and relaxation exercises can also reduce anxiety.",
    'surprise': "Surprise can be exciting or overwhelming. Take a moment to process the unexpected event. If it's a negative surprise, lean on your support system.",
    'disgust': "Explore the source of your disgust. If it's related to a situation or behavior, consider setting boundaries or making changes. If it's related to food or sensory input, explore coping mechanisms.",
    'neutral': "A neutral state is a good baseline. Continue engaging in activities that support your overall well-being.",
    # Add more emotions and suggestions as needed
}


# Function to analyze text
def analyze_text(text):
    if not sentiment_model or not emotion_model:
        return text, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available."
    sentiment = sentiment_model(text)[0]
    emotion = emotion_model(text)[0]
    emotion_label = emotion['label']
    therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
    return (
        text,
        sentiment['label'], round(sentiment['score'], 2),
        emotion_label, round(emotion['score'], 2),
        therapy_suggestion
    )

# Function to analyze audio
def analyze_audio(audio_file):
    if not whisper_model or not sentiment_model or not emotion_model:
         return audio_file, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available."
    # Transcribe audio to text
    result = whisper_model.transcribe(audio_file)
    text = result["text"]
    # Analyze the transcribed text
    text_analysis_results = analyze_text(text)
    # The analyze_text function now returns the therapy suggestion as the last element
    return (audio_file,) + text_analysis_results[1:] # Keep the audio file input and append text analysis results

from transformers import pipeline
from google.colab import files
import os

try:
    # Image emotion detection model
    image_emotion_model = pipeline(
        "image-classification",
        model="trpakov/vit-face-expression",  # or "dima806/facial_emotions_image_detection"
        device=device
    )
    print("✅ Image emotion model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load image emotion model: {e}")
    image_emotion_model = None

def analyze_image(image_path):
    print(f"Attempting to analyze image at path: {image_path}")
    if not image_emotion_model:
        print("Image model not loaded.")
        return image_path, "Image model not loaded. Please check the logs.", "Therapy suggestion not available."

    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
        return image_path, f"❌ Error: Image file not found at {image_path}", "Therapy suggestion not available."

    try:
        results = image_emotion_model(image_path)
        print(f"Image analysis results: {results}")
        # Sort results by confidence and get the top emotion
        top_emotion = sorted(results, key=lambda x: x['score'], reverse=True)[0]
        emotion_label = top_emotion['label']
        score = round(top_emotion['score'], 2)
        therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
        return image_path, f"🖼️ Detected emotion: {emotion_label} (Confidence: {score})", therapy_suggestion
    except Exception as e:
        print(f"❌ Failed to analyze image: {e}")
        return image_path, f"❌ Failed to analyze image: {e}", "Therapy suggestion not available."


# Add code to upload an audio file for testing
def upload_audio():
  uploaded = files.upload()
  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    return fn # Return the filename of the uploaded file

if __name__ == "__main__":
    # Example: analyze text
    print(analyze_text("I’m very happy to see this project working!"))

    # Example: analyze audio
    # Upload an audio file before running this line
    # audio_file_name = upload_audio()
    # if audio_file_name:
    #     print(analyze_audio(audio_file_name))
    # else:
    #     print("No audio file uploaded.")

    # Example: analyze image
    # Upload an image file before running this line
    # image_file_name = upload_audio() # This should be upload_image()
    # if image_file_name:
    #     print(analyze_image(image_file_name))
    # else:
    #     print("No image file uploaded.")
    pass # Keeping the original code structure but commenting out the failing parts

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


🔄 Loading models... This may take a minute.
Using device: cpu


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Sentiment model loaded successfully!


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ Image emotion model loaded successfully!
('I’m very happy to see this project working!', 'POSITIVE', 1.0, 'joy', 0.99, 'Consider celebrating your positive emotions! Sharing your joy with others or engaging in activities that make you happy can enhance well-being.')


In [5]:
# ================================
# Creating the Gradio Interface
# ================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎯 Sentiment & Emotion Detector
        **Analyze text, audio, or images to detect sentiment and emotion.**
        Powered by **Hugging Face Transformers** and **OpenAI Whisper**.
        Supports **multilingual input** (audio auto-detected).
        """
    )

    with gr.Tab("📝 Text Input"):
        text_input = gr.Textbox(lines=3, placeholder="Type your text here...")
        sentiment_label = gr.Label(label="Sentiment")
        sentiment_conf = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label = gr.Label(label="Emotion")
        emotion_conf = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        text_button = gr.Button("Analyze Text")

        text_button.click(
            analyze_text,
            inputs=text_input,
            outputs=[text_input, sentiment_label, sentiment_conf, emotion_label, emotion_conf]
        )

    with gr.Tab("🎤 Audio Input"):
        audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Audio")
        sentiment_label_a = gr.Label(label="Sentiment")
        sentiment_conf_a = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label_a = gr.Label(label="Emotion")
        emotion_conf_a = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        audio_button = gr.Button("Analyze Audio")

        audio_button.click(
            analyze_audio,
            inputs=audio_input,
            outputs=[audio_input, sentiment_label_a, sentiment_conf_a, emotion_label_a, emotion_conf_a]
        )

    with gr.Tab("🖼️ Image Input"):
        image_input = gr.Image(type="filepath", label="Upload Image")
        image_emotion_output = gr.Textbox(label="Image Analysis Result")
        image_button = gr.Button("Analyze Image")

        image_button.click(
            analyze_image,
            inputs=image_input,
            outputs=image_emotion_output
        )

/tmp/ipykernel_43030/2025440949.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [6]:
# Define a mapping of emotions to therapy suggestions
emotion_therapies = {
    'joy': "Consider celebrating your positive emotions! Sharing your joy with others or engaging in activities that make you happy can enhance well-being.",
    'sadness': "It's okay to feel sad. Talking to a friend, family member, or therapist can help. Engaging in self-care activities like mindfulness or gentle exercise can also be beneficial.",
    'anger': "Finding healthy ways to manage anger is important. Techniques like deep breathing, journaling, or physical activity can help. If anger is persistent, consider talking to a professional.",
    'fear': "Facing fears can be challenging, but also empowering. Techniques like gradual exposure or cognitive restructuring with a therapist can be helpful. Mindfulness and relaxation exercises can also reduce anxiety.",
    'surprise': "Surprise can be exciting or overwhelming. Take a moment to process the unexpected event. If it's a negative surprise, lean on your support system.",
    'disgust': "Explore the source of your disgust. If it's related to a situation or behavior, consider setting boundaries or making changes. If it's related to food or sensory input, explore coping mechanisms.",
    'neutral': "A neutral state is a good baseline. Continue engaging in activities that support your overall well-being.",
    # Add more emotions and suggestions as needed
}

# Function to analyze text
def analyze_text(text):
    if not sentiment_model or not emotion_model:
        return text, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available."
    sentiment = sentiment_model(text)[0]
    emotion = emotion_model(text)[0]
    emotion_label = emotion['label']
    therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
    return (
        text,
        sentiment['label'], round(sentiment['score'], 2),
        emotion_label, round(emotion['score'], 2),
        therapy_suggestion
    )

# Function to analyze audio
def analyze_audio(audio_file):
    if not whisper_model or not sentiment_model or not emotion_model:
         return audio_file, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available."
    # Transcribe audio to text
    result = whisper_model.transcribe(audio_file)
    text = result["text"]
    # Analyze the transcribed text
    text_analysis_results = analyze_text(text)
    # The analyze_text function now returns the therapy suggestion as the last element
    return (audio_file,) + text_analysis_results[1:] # Keep the audio file input and append text analysis results

# Function to analyze image
def analyze_image(image_path):
    print(f"Attempting to analyze image at path: {image_path}")
    if not image_emotion_model:
        print("Image model not loaded.")
        return image_path, "Image model not loaded. Please check the logs.", "Therapy suggestion not available."

    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
        return image_path, f"❌ Error: Image file not found at {image_path}", "Therapy suggestion not available."

    try:
        results = image_emotion_model(image_path)
        print(f"Image analysis results: {results}")
        # Sort results by confidence and get the top emotion
        top_emotion = sorted(results, key=lambda x: x['score'], reverse=True)[0]
        emotion_label = top_emotion['label']
        score = round(top_emotion['score'], 2)
        therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
        return image_path, f"🖼️ Detected emotion: {emotion_label} (Confidence: {score})", therapy_suggestion
    except Exception as e:
        print(f"❌ Failed to analyze image: {e}")
        return image_path, f"❌ Failed to analyze image: {e}", "Therapy suggestion not available."


In [7]:
# ================================
# Creating the Gradio Interface
# ================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎯 Sentiment & Emotion Detector
        **Analyze text, audio, or images to detect sentiment and emotion.**
        Powered by **Hugging Face Transformers** and **OpenAI Whisper**.
        Supports **multilingual input** (audio auto-detected).
        """
    )

    with gr.Tab("📝 Text Input"):
        text_input = gr.Textbox(lines=3, placeholder="Type your text here...")
        sentiment_label = gr.Label(label="Sentiment")
        sentiment_conf = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label = gr.Label(label="Emotion")
        emotion_conf = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        text_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3) # Add therapy output for text
        text_button = gr.Button("Analyze Text")

        text_button.click(
            analyze_text,
            inputs=text_input,
            outputs=[text_input, sentiment_label, sentiment_conf, emotion_label, emotion_conf, text_therapy_output] # Update outputs
        )

    with gr.Tab("🎤 Audio Input"):
        audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Audio")
        sentiment_label_a = gr.Label(label="Sentiment")
        sentiment_conf_a = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label_a = gr.Label(label="Emotion")
        emotion_conf_a = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        audio_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3) # Add therapy output for audio
        audio_button = gr.Button("Analyze Audio")

        audio_button.click(
            analyze_audio,
            inputs=audio_input,
            outputs=[audio_input, sentiment_label_a, sentiment_conf_a, emotion_label_a, emotion_conf_a, audio_therapy_output] # Update outputs
        )

    with gr.Tab("🖼️ Image Input"):
        image_input = gr.Image(type="filepath", label="Upload Image")
        image_emotion_output = gr.Textbox(label="Image Analysis Result", interactive=False) # Make this interactive=False
        image_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3) # Add therapy output for image
        image_button = gr.Button("Analyze Image")

        image_button.click(
            analyze_image,
            inputs=image_input,
            outputs=[image_input, image_emotion_output, image_therapy_output] # Update outputs
        )


/tmp/ipykernel_43030/3600538606.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [8]:
emotion_songs = {
    'joy': [
        "https://open.spotify.com/track/0V6XoG8tWC7xhUNwYLOGYL?si=fba681b7218e465c",
        "https://open.spotify.com/track/5n1FBtJgcDePfo8q6hBQPu?si=87cc9c8805434755",
        "https://open.spotify.com/track/5mBwJ0Dl4GSoXwbATPtcsj?si=1c8e2c91aa6c41e3",
        "https://open.spotify.com/track/6QQyi87suKD8l4WcNjwO8K?si=22e8e306149c4b09"
    ],
    'sad': [
        "https://open.spotify.com/track/25gEVFRPKTn0QLlZmQPYtQ?si=58e1d173d50c4c21",
        "https://open.spotify.com/track/34ofcNiZg3Ozj90bHCt6wu?si=316a6361b3944392",
        "https://open.spotify.com/track/4R5E554SteM6KB6bLHTU29?si=00105b2cede24e1d",
        "https://open.spotify.com/track/5Mo0zkOAPtR2rW9kL29X37?si=2923462f51c246e2"
    ],
    'anger': [
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=6bc22f9f0f8c4736",
        "https://open.spotify.com/track/1mHEvHzpRwCLeAnecvE6eS?si=0319aabec8b8481a",
        "https://open.spotify.com/track/48cfDmvuqRAC9cXdYJqNMM?si=383c55f1b3be486b",
        "https://open.spotify.com/track/0BAVYSf6ghAs5V3DfuLHDu?si=6ddf6e525cc24e95"
    ],
    'fear': [
        "https://open.spotify.com/track/7o3gnrSdBN1yJeyDd3ysWX?si=d6aec237ecdf4d73",
        "https://open.spotify.com/track/2t0rnpPY1ZdfguwbGinqXj?si=756a0be2135b4e4a",
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=a6b141fa7cc141e1"

    ],
    'surprise': [
        "https://open.spotify.com/track/2tudwIWp4v2VwGgQoJc4Q3", # Bohemian Rhapsody - Queen
        "https://open.spotify.com/track/2eejB9BvQ7P8c7c1b80Vz8", # Smells Like Teen Spirit - Nirvana
        "https://open.spotify.com/track/6habFhs2twptgc5gYhshb"  # Billie Jean - Michael Jackson
    ],
    'disgust': [
        "https://open.spotify.com/track/3bidbhpCiNvO1femzXHJP", # Creep - Radiohead
        "https://open.spotify.com/track/0OaM08r0bJ5R1B4zR0M4Ew", # Closer - Nine Inch Nails
        "https://open.spotify.com/track/2tDud0sQ8tM3S8mJjI3M0V"  # Hurt - Nine Inch Nails
    ],
    'neutral': [
        "https://open.spotify.com/track/2PZEMlGRrX9s7tA5H5Q4wZ", # Weightless - Marconi Union
        "https://open.spotify.com/track/7fPuWRLxYl6g53fO64eozf", # Ambient 1: Music for Airports - Brian Eno
        "https://open.spotify.com/track/6t6unw1J3mS3KzX2k1E2g"  # Gymnopedie No. 1 - Erik Satie
    ]
}

In [9]:
import random

# Define a mapping of emotions to therapy suggestions
emotion_therapies = {
    'joy': "Consider celebrating your positive emotions! Sharing your joy with others or engaging in activities that make you happy can enhance well-being.",
    'sadness': "It's okay to feel sad. Talking to a friend, family member, or therapist can help. Engaging in self-care activities like mindfulness or gentle exercise can also be beneficial.",
    'anger': "Finding healthy ways to manage anger is important. Techniques like deep breathing, journaling, or physical activity can help. If anger is persistent, consider talking to a professional.",
    'fear': "Facing fears can be challenging, but also empowering. Techniques like gradual exposure or cognitive restructuring with a therapist can be helpful. Mindfulness and relaxation exercises can also reduce anxiety.",
    'surprise': "Surprise can be exciting or overwhelming. Take a moment to process the unexpected event. If it's a negative surprise, lean on your support system.",
    'disgust': "Explore the source of your disgust. If it's related to a situation or behavior, consider setting boundaries or making changes. If it's related to food or sensory input, explore coping mechanisms.",
    'neutral': "A neutral state is a good baseline. Continue engaging in activities that support your overall well-being."
}

# Define a mapping of emotions to Spotify song links
emotion_songs = {
    'joy': [
        "https://open.spotify.com/track/0V6XoG8tWC7xhUNwYLOGYL?si=fba681b7218e465c",
        "https://open.spotify.com/track/5n1FBtJgcDePfo8q6hBQPu?si=87cc9c8805434755",
        "https://open.spotify.com/track/5mBwJ0Dl4GSoXwbATPtcsj?si=1c8e2c91aa6c41e3",
        "https://open.spotify.com/track/6QQyi87suKD8l4WcNjwO8K?si=22e8e306149c4b09"
    ],
    'sadness': [
        "https://open.spotify.com/track/25gEVFRPKTn0QLlZmQPYtQ?si=58e1d173d50c4c21",
        "https://open.spotify.com/track/34ofcNiZg3Ozj90bHCt6wu?si=316a6361b3944392",
        "https://open.spotify.com/track/4R5E554SteM6KB6bLHTU29?si=00105b2cede24e1d",
        "https://open.spotify.com/track/5Mo0zkOAPtR2rW9kL29X37?si=2923462f51c246e2"
    ],
    'anger': [
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=6bc22f9f0f8c4736",
        "https://open.spotify.com/track/1mHEvHzpRwCLeAnecvE6eS?si=0319aabec8b8481a",
        "https://open.spotify.com/track/48cfDmvuqRAC9cXdYJqNMM?si=383c55f1b3be486b",
        "https://open.spotify.com/track/0BAVYSf6ghAs5V3DfuLHDu?si=6ddf6e525cc2e95"
    ],
    'fear': [
        "https://open.spotify.com/track/7o3gnrSdBN1yJeyDd3ysWX?si=d6aec237ecdf4d73",
        "https://open.spotify.com/track/2t0rnpPY1ZdfguwbGinqXj?si=756a0be2135b4e4a",
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=a6b141fa7cc141e1"

    ],
    'surprise': [
        "https://open.spotify.com/track/2tudwIWp4v2VwGgQoJcQ3",
        "https://open.spotify.com/track/2eejB9BvQ7P8c7c1b80Vz8",
        "https://open.spotify.com/track/6habFhs2twptgc5gYhshb"
    ],
    'disgust': [
        "https://open.spotify.com/track/3bidbhpCiNvO1femzXHJP",
        "https://open.spotify.com/track/0OaM08r0bJ5R1B4zR0M4Ew",
        "https://open.spotify.com/track/2tDud0sQ8tM3S8mJjI3M0V"
    ],
    'neutral': [
        "https://open.spotify.com/track/2PZEMlGRrX9s7tA5nH5Q4wZ",
        "https://open.spotify.com/track/7fPuWRLxYl6g53fO64eozf",
        "https://open.spotify.com/track/6t6unw1J3mS3KzX2k1E2g"
    ]
}

# Function to generate Spotify embed code
def get_spotify_embed_code(track_url):
    print(f"Generating embed code for URL: {track_url}")
    if "open.spotify.com/track/" in track_url:
        track_id = track_url.split("/")[-1].split("?")[0]
        embed_code = f'<iframe src="https://open.spotify.com/embed/track/{track_id}" width="300" height="80" frameborder="0" allowtransparency="true" allow="encrypted-media"></iframe>'
        print(f"Generated embed code: {embed_code}")
        return embed_code
    print("Invalid Spotify track URL.")
    return "Invalid Spotify track URL."


# Function to analyze text
def analyze_text(text):
    if not sentiment_model or not emotion_model:
        return text, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available.", "Song suggestion not available."
    sentiment = sentiment_model(text)[0]
    emotion = emotion_model(text)[0]
    emotion_label = emotion['label'].lower().strip() # Add .strip() here
    therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
    song_url = random.choice(emotion_songs.get(emotion_label, ["No song suggestion available."])) if emotion_songs.get(emotion_label) else "No song suggestion available."
    song_suggestion_html = get_spotify_embed_code(song_url) if song_url and song_url.startswith("http") else song_url
    return (
        text,
        sentiment['label'], round(sentiment['score'], 2),
        emotion_label, round(emotion['score'], 2),
        therapy_suggestion,
        song_suggestion_html
    )

# Function to analyze audio
def analyze_audio(audio_file):
    if not whisper_model or not sentiment_model or not emotion_model:
         return audio_file, "Models not loaded. Please check the logs.", 0, "Models not loaded. Please check the logs.", 0, "Therapy suggestion not available.", "Song suggestion not available."
    # Transcribe audio to text
    result = whisper_model.transcribe(audio_file)
    text = result["text"]
    # Analyze the transcribed text
    text_analysis_results = analyze_text(text)
    # The analyze_text function now returns the therapy suggestion and song suggestion as the last elements
    return (audio_file,) + text_analysis_results[1:]

# Function to analyze image
def analyze_image(image_path):
    print(f"Attempting to analyze image at path: {image_path}")
    if not image_emotion_model:
        print("Image model not loaded.")
        return image_path, "Image model not loaded. Please check the logs.", "Therapy suggestion not available.", "Song suggestion not available."

    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
        return image_path, f"❌ Error: Image file not found at {image_path}", "Therapy suggestion not available.", "Song suggestion not available."

    try:
        results = image_emotion_model(image_path)
        print(f"Image analysis results: {results}")
        # Sort results by confidence and get the top emotion
        top_emotion = sorted(results, key=lambda x: x['score'], reverse=True)[0]
        emotion_label = top_emotion['label'].lower().strip() # Add .strip() here
        score = round(top_emotion['score'], 2)
        print(f"Detected image emotion label (lowercase): {emotion_label}")

        therapy_suggestion = emotion_therapies.get(emotion_label, "No specific therapy suggestion for this emotion.")
        song_url = random.choice(emotion_songs.get(emotion_label, ["No song suggestion available."])) if emotion_songs.get(emotion_label) else "No song suggestion available."
        song_suggestion_html = get_spotify_embed_code(song_url) if song_url and song_url.startswith("http") else song_url
        return image_path, f"🖼️ Detected emotion: {emotion_label.capitalize()} (Confidence: {score})", therapy_suggestion, song_suggestion_html
    except Exception as e:
        print(f"❌ Failed to analyze image: {e}")
        return image_path, f"❌ Failed to analyze image: {e}", "Therapy suggestion not available.", "Song suggestion not available."


In [10]:
# ================================
# Creating the Gradio Interface
# ================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎯 Sentiment & Emotion Detector
        **Analyze text, audio, or images to detect sentiment and emotion.**
        Powered by **Hugging Face Transformers** and **OpenAI Whisper**.
        Supports **multilingual input** (audio auto-detected).
        """
    )

    with gr.Tab("📝 Text Input"):
        text_input = gr.Textbox(lines=3, placeholder="Type your text here...")
        sentiment_label = gr.Label(label="Sentiment")
        sentiment_conf = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label = gr.Label(label="Emotion")
        emotion_conf = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        text_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        text_song_output = gr.Textbox(label="Song Suggestion", interactive=False, lines=1) # Reverted to Textbox
        text_button = gr.Button("Analyze Text")

        text_button.click(
            analyze_text,
            inputs=text_input,
            outputs=[text_input, sentiment_label, sentiment_conf, emotion_label, emotion_conf, text_therapy_output, text_song_output]
        )

    with gr.Tab("🎤 Audio Input"):
        audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Audio")
        sentiment_label_a = gr.Label(label="Sentiment")
        sentiment_conf_a = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label_a = gr.Label(label="Emotion")
        emotion_conf_a = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        audio_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        audio_song_output = gr.Textbox(label="Song Suggestion", interactive=False, lines=1) # Reverted to Textbox
        audio_button = gr.Button("Analyze Audio")

        audio_button.click(
            analyze_audio,
            inputs=audio_input,
            outputs=[audio_input, sentiment_label_a, sentiment_conf_a, emotion_label_a, emotion_conf_a, audio_therapy_output, audio_song_output]
        )

    with gr.Tab("🖼️ Image Input"):
        image_input = gr.Image(type="filepath", label="Upload Image")
        image_emotion_output = gr.Textbox(label="Image Analysis Result", interactive=False)
        image_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        image_song_output = gr.Textbox(label="Song Suggestion", interactive=False, lines=1) # Reverted to Textbox
        image_button = gr.Button("Analyze Image")

        image_button.click(
            analyze_image,
            inputs=image_input,
            outputs=[image_input, image_emotion_output, image_therapy_output, image_song_output]
        )

/tmp/ipykernel_43030/3501530759.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [11]:
import gradio as gr # Ensure gradio is imported here as well

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎯 Sentiment & Emotion Detector
        **Analyze text, audio, or images to detect sentiment and emotion.**
        Powered by **Hugging Face Transformers** and **OpenAI Whisper**.
        Supports **multilingual input** (audio auto-detected).
        """
    )

    with gr.Tab("📝 Text Input"):
        text_input = gr.Textbox(lines=3, placeholder="Type your text here...")
        sentiment_label = gr.Label(label="Sentiment")
        sentiment_conf = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label = gr.Label(label="Emotion")
        emotion_conf = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        text_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        text_song_output = gr.HTML(label="Song Suggestion") # Change to gr.HTML
        text_button = gr.Button("Analyze Text")

        text_button.click(
            analyze_text,
            inputs=text_input,
            outputs=[text_input, sentiment_label, sentiment_conf, emotion_label, emotion_conf, text_therapy_output, text_song_output]
        )

    with gr.Tab("🎤 Audio Input"):
        audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Audio")
        sentiment_label_a = gr.Label(label="Sentiment")
        sentiment_conf_a = gr.Slider(0, 1, label="Sentiment Confidence", interactive=False)
        emotion_label_a = gr.Label(label="Emotion")
        emotion_conf_a = gr.Slider(0, 1, label="Emotion Confidence", interactive=False)
        audio_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        audio_song_output = gr.HTML(label="Song Suggestion") # Change to gr.HTML
        audio_button = gr.Button("Analyze Audio")

        audio_button.click(
            analyze_audio,
            inputs=audio_input,
            outputs=[audio_input, sentiment_label_a, sentiment_conf_a, emotion_label_a, emotion_conf_a, audio_therapy_output, audio_song_output]
        )

    with gr.Tab("🖼️ Image Input"):
        image_input = gr.Image(sources=["upload", "webcam"], type="filepath", label="Upload Image or Use Webcam") # Changed to include webcam
        image_emotion_output = gr.Textbox(label="Image Analysis Result", interactive=False)
        image_therapy_output = gr.Textbox(label="Therapy Suggestion", interactive=False, lines=3)
        image_song_output = gr.HTML(label="Song Suggestion") # Change to gr.HTML
        image_button = gr.Button("Analyze Image")

        image_button.click(
            analyze_image,
            inputs=image_input,
            outputs=[image_input, image_emotion_output, image_therapy_output, image_song_output]
        )

/tmp/ipykernel_43030/3437793860.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


# Task
Update the emotion mappings in `emotion_therapies` and `emotion_songs` dictionaries to match the exact lowercase emotion labels output by the emotion detection model (e.g., 'sadness', 'joy'), remove redundant entries, and then re-run and launch the Gradio interface.

## Update Emotion Mappings

### Subtask:
Modify the `emotion_therapies` and `emotion_songs` dictionaries to ensure their keys (`sad`, `happy`, `Sad`) exactly match the lowercase emotion labels (`sadness`, `joy`) output by the emotion detection model.


**Reasoning**:
The subtask requires updating the `emotion_therapies` and `emotion_songs` dictionaries to standardize their keys according to the instructions, ensuring consistency with the model's lowercase output. This involves removing redundant 'happy' keys and changing 'Sad' and 'sad' to 'sadness' while retaining their associated values.



In [12]:
import random

# Define a mapping of emotions to therapy suggestions
emotion_therapies = {
    'joy': "Consider celebrating your positive emotions! Sharing your joy with others or engaging in activities that make you happy can enhance well-being.",
    'sadness': "It's okay to feel sad. Talking to a friend, family member, or therapist can help. Engaging in self-care activities like mindfulness or gentle exercise can also be beneficial.", # Changed 'Sad' to 'sadness'
    'anger': "Finding healthy ways to manage anger is important. Techniques like deep breathing, journaling, or physical activity can help. If anger is persistent, consider talking to a professional.",
    'fear': "Facing fears can be challenging, but also empowering. Techniques like gradual exposure or cognitive restructuring with a therapist can be helpful. Mindfulness and relaxation exercises can also reduce anxiety.",
    'surprise': "Surprise can be exciting or overwhelming. Take a moment to process the unexpected event. If it's a negative surprise, lean on your support system.",
    'disgust': "Explore the source of your disgust. If it's related to a situation or behavior, consider setting boundaries or making changes. If it's related to food or sensory input, explore coping mechanisms.",
    'neutral': "A neutral state is a good baseline. Continue engaging in activities that support your overall well-being.",
    # Removed redundant 'happy' key
}

# Define a mapping of emotions to Spotify song links
emotion_songs = {
    'joy': [
        "https://open.spotify.com/track/0V6XoG8tWC7xhUNwYLOGYL?si=fba681b7218e465c",
        "https://open.spotify.com/track/5n1FBtJgcDePfo8q6hBQPu?si=87cc9c8805434755",
        "https://open.spotify.com/track/5mBwJ0Dl4GSoXwbATPtcsj?si=1c8e2c91aa6c41e3",
        "https://open.spotify.com/track/6QQyi87suKD8l4WcNjwO8K?si=22e8e306149c4b09"
    ],
    'sadness': [
        "https://open.spotify.com/track/25gEVFRPKTn0QLlZmQPYtQ?si=58e1d173d50c4c21",
        "https://open.spotify.com/track/34ofcNiZg3Ozj90bHCt6wu?si=316a6361b3944392",
        "https://open.spotify.com/track/4R5E554SteM6KB6bLHTU29?si=00105b2cede24e1d",
        "https://open.spotify.com/track/5Mo0zkOAPtR2rW9kL29X37?si=2923462f51c246e2"
    ], # Changed 'sad' to 'sadness', removed redundant 'happy' key
    'anger': [
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=6bc22f9f0f8c4736",
        "https://open.spotify.com/track/1mHEvHzpRwCLeAnecvE6eS?si=0319aabec8b8481a",
        "https://open.spotify.com/track/48cfDmvuqRAC9cXdYJqNMM?si=383c55f1b3be486b",
        "https://open.spotify.com/track/0BAVYSf6ghAs5V3DfuLHDu?si=6ddf6e525cc2e95"
    ],
    'fear': [
        "https://open.spotify.com/track/7o3gnrSdBN1yJeyDd3ysWX?si=d6aec237ecdf4d73",
        "https://open.spotify.com/track/2t0rnpPY1ZdfguwbGinqXj?si=756a0be2135b4e4a",
        "https://open.spotify.com/track/3lLX2Vm99sMD5kuZqnr2T0?si=a6b141fa7cc141e1"

    ],
    'surprise': [
        "https://open.spotify.com/track/2tudwIWp4v2VwGgQoJcQ3", # Bohemian Rhapsody - Queen
        "https://open.spotify.com/track/2eejB9BvQ7P8c7c1b80Vz8", # Smells Like Teen Spirit - Nirvana
        "https://open.spotify.com/track/6habFhs2twptgc5gYhshb"  # Billie Jean - Michael Jackson
    ],
    'disgust': [
        "https://open.spotify.com/track/3bidbhpCiNvO1femzXHJP", # Creep - Radiohead
        "https://open.spotify.com/track/0OaM08r0bJ5R1B4zR0M4Ew", # Closer - Nine Inch Nails
        "https://open.spotify.com/track/2tDud0sQ8tM3S8mJjI3M0V"  # Hurt - Nine Inch Nails
    ],
    'neutral': [
        "https://open.spotify.com/track/2PZEMlGRrX9s7tA5nH5Q4wZ", # Weightless - Marconi Union
        "https://open.spotify.com/track/7fPuWRLxYl6g53fO64eozf", # Ambient 1: Music for Airports - Brian Eno
        "https://open.spotify.com/track/6t6unw1J3mS3KzX2k1E2g"  # Gymnopedie No. 1 - Erik Satie
    ]
}

# The functions below will use the updated dictionaries
print("Emotion mappings and song suggestions updated.")

Emotion mappings and song suggestions updated.


**Reasoning**:
The dictionaries have been updated, and now the Gradio interface needs to be launched again to reflect these changes and enable the new song suggestion feature.



In [13]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a72c8615c73b1eaa65.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
